In [1]:
import os
import glob
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import json
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm

# collect scores

In [41]:
# Base directory containing all expression runs
base_dir = "../../runs/expression/qnorm/qnorm_v1/"
# base_dir = "/runs/expression" # for mb machine

# Function to extract scores from TensorBoard event files
def extract_scores_from_events(event_file_path, run_name):
	"""Extract all scalar values at a specific step from TensorBoard event file."""
	try:
		ea = EventAccumulator(event_file_path)
		ea.Reload()
		
		scores = []
		for tag in ea.Tags()['scalars']:
			events = ea.Scalars(tag)
			# Find events at or closest to target step
			step_values = [(event.step, event.value) for event in events]
						
			for step, value in step_values:
				if step > 100: # skip test runs
					scores.append({"step": step, "value": value, "run_name": run_name, "metric": tag})
		return scores
	except Exception as e:
		print(f"Error processing {event_file_path}: {e}")
		return {}

# Main function to traverse all subdirectories and collect scores
def collect_all_scores(base_dir):
    """Traverse all subdirectories recursively and collect scores at specified step."""
    all_results = []
    all_dirs = list(os.walk(base_dir))
    for ind, (root, dirs, files) in enumerate(all_dirs):
        # Find TensorBoard event files in this directory
        event_files = glob.glob(os.path.join(root, "events.out.tfevents*"))
        if not event_files:
            continue
        print (ind, " of ",len(all_dirs))            
        # print(f"Processing: {root}")
        for event_file in event_files:
            # Only keep the part of the path after base_dir in run_name
            rel_run_name = os.path.relpath(root, base_dir)
            scores = extract_scores_from_events(event_file, run_name=rel_run_name)
            if scores:
                all_results.extend(scores)
                # print(f"  Found {len(scores)} metrics")
            else:
                pass
                # print(f"  No scores found")
    return all_results
	
# Execute the collection
print(f"Starting to collect scores from all subdirectories...")
results = collect_all_scores(base_dir)

print(f"\nCollected data from {len(results)} runs")

# Convert to DataFrame for easier analysis
if results:
	df = pd.DataFrame.from_records(results)
	print(f"\nDataFrame shape: {df.shape}")
	print("\nColumns:", df.columns.tolist())
	
	# Display first few rows
	print("\nFirst few results:")
	display(df.head())
	
	# Save to CSV
	output_file = f"expression_scores.csv"
	df.to_csv(output_file, index=False, compression="gzip")
	print(f"\nResults saved to {output_file}")
else:
	print("No results found!")

Starting to collect scores from all subdirectories...
1  of  8
5  of  8

Collected data from 270912 runs

DataFrame shape: (270912, 4)

Columns: ['step', 'value', 'run_name', 'metric']

First few results:


,step,value,run_name,metric
0,150,3.122101,20250710-153301,loss/iterations/train
1,200,2.902649,20250710-153301,loss/iterations/train
2,250,2.862545,20250710-153301,loss/iterations/train
3,300,2.611175,20250710-153301,loss/iterations/train
4,350,2.554961,20250710-153301,loss/iterations/train



Results saved to expression_scores.csv


In [3]:
df["run_name"].unique()

array(['one-hot', 'logemb', 'megafull', 'upsample.ctloss.megafull',
       'upsample.ctloss_nonorm.megafull'], dtype=object)

# Output scores for selected run in table-ready format

In [5]:
df = pd.read_csv("expression_scores.csv", compression="gzip")
# df = pd.read_csv("expression_scores_25-07-2025.csv", compression="gzip")
df.head(2)

,step,value,run_name,metric
0,150,3.122101,qnorm_v1/20250710-153301,loss/iterations/train
1,200,2.902649,qnorm_v1/20250710-153301,loss/iterations/train


In [43]:
df["run_name"].unique()

array(['20250710-153301',
       'resume_20250710-153301/model_80000.pth/20250714-172332'],
      dtype=object)

In [50]:
selected_run = 'resume_20250710-153301/model_80000.pth/20250714-172332'

In [53]:
sel = df.query("run_name == @selected_run")
_met = [i for i in sel.metric.unique() if ("valid" in i) and ("v1" in i or "loss" in i) and ("samples" in i) and not ("_4_" in i)]
sel = sel.query("metric in @_met").dropna(subset=["value"])
results = sel.groupby("metric").apply(lambda x: x.loc[x["value"].idxmin()] if x["metric"].str.contains("loss").any() else x.loc[x["value"].idxmax()])
results.reset_index(drop=True).sort_values("metric", ascending=False)

,step,value,run_name,metric
9,4015550,0.377212,resume_20250710-153301/model_80000.pth/2025071...,score_predictions_Expression_dataset_v1_mm10_C...
8,3995950,0.415331,resume_20250710-153301/model_80000.pth/2025071...,score_predictions_Expression_dataset_v1_mm10_C...
7,4958800,0.068408,resume_20250710-153301/model_80000.pth/2025071...,score_predictions_Expression_dataset_v1_GRCh38...
6,5556600,0.132398,resume_20250710-153301/model_80000.pth/2025071...,score_predictions_Expression_dataset_v1_GRCh38...
5,4574150,0.659233,resume_20250710-153301/model_80000.pth/2025071...,pearson_corr_genes_Expression_dataset_v1_mm10_...
4,4206650,0.667348,resume_20250710-153301/model_80000.pth/2025071...,pearson_corr_genes_Expression_dataset_v1_GRCh3...
3,4713800,0.079532,resume_20250710-153301/model_80000.pth/2025071...,pearson_corr_cells_Expression_dataset_v1_mm10_...
2,10091550,0.108385,resume_20250710-153301/model_80000.pth/2025071...,pearson_corr_cells_Expression_dataset_v1_GRCh3...
1,4206650,1.704311,resume_20250710-153301/model_80000.pth/2025071...,loss_cls/samples/valid
0,4206650,1.704311,resume_20250710-153301/model_80000.pth/2025071...,loss/samples/valid


In [54]:
# select steps when scpre prediction on v1 is best (2 steps probably, one for human, one for mouse)
steps = results[results.metric.str.startswith('score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid [slow-10]')]["step"].values

In [56]:
# select steps when scpre prediction on v1 is best (2 steps probably, one for human, one for mouse)
steps = results[results.metric.str.startswith('score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid [slow-10]')]["step"].values

In [57]:
for step in steps:
	# sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)
	_met_sorted = [
		 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid [slow-10]' ,
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid [slow-10]',
  'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
  'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
   'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
]
	print (selected_run, end="\t")
	print (step, end="\t")
	for met in _met_sorted:
		# print (met, end="\t")
		v = sel.query("metric == @met and step == @step")["value"].values[0]
		print (v, end="\t")
	print("\n------")

resume_20250710-153301/model_80000.pth/20250714-172332	4015550	-0.05968768894672394	0.37721240520477295	0.057442449033260345	0.05514390766620636	0.657572865486145	0.6559485793113708	
------


In [32]:
for step in steps:
	# sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)
	_met_sorted = [
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid [slow-10]' ,
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
]
	print (selected_run, end="\t")
	print (step, end="\t")
	for met in _met_sorted:
		# print (met, end="\t")
		v = sel.query("metric == @met and step == @step")["value"].values[0]
		print (v, end="\t")
	print("\n------")

qnorm_v1_human_large_1seg_quantloss/20250819-000423	617400	0.016463052481412888	0.12176636606454849	0.6644256114959717	
------


In [10]:
for step in steps:
	# sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)
	_met_sorted = [
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid [slow-10]',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
]
	print (selected_run, end="\t")
	print (step, end="\t")
	for met in _met_sorted:
		# print (met, end="\t")
		v = sel.query("metric == @met and step == @step")["value"].values[0]
		print (v, end="\t")
	print("\n------")

megafull	27596800	0.04793844372034073	0.018366871401667595	0.6495351195335388	
------


44800
0.1085683479905128	0.110475406050682	-0.0988152027130127	-0.0105760768055915	0.5893904566764832	0.5265839695930481	

In [62]:
_met

['loss/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid']

In [ ]:
import sys
sys.path.append("/home/jovyan/dnalm/downstream_tasks/annotation")

from transcript_GenePredictionDataset_letter_level_CADUSEUS_intergenic import AnnotationDataset
./data/tokenizers/t2t_1000h_multi_32k/
dataset = AnnotationDataset("/home/jovyan/shares/SR003.nfs2/chr_dataset_human/chr_dataset_gena_gpt_valid.hdf5", "/home/jovyan/shares/SR003.nfs2/chr_dataset_human/chr_dataset_gena_gpt_test.hdf5")

for i in range(len(dataset)):
    print(dataset[i])
    break

/workspace-SR003.nfs2/aspeedok/aspeedok/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: __init__() missing 2 required positional arguments: 'tokenizer' and 'tokenizer_caduseus'